# DDL Script : Create Gold View 

## Script Purpose :
-   This script create the views for the gold layer in the data ware house.
-   The Gold Layer represents the final dimension and fact tables (Star Schema)
-   
-   EA view performs transformations and combines data from the silver layer to product a clean , enriched and business-ready dataset.

## Usage :
-   These views can be queried directly for analytics and reporting 


In [0]:
#Build the star schema like below 
#             dim_customers
#                    │
#                    │
#dim_products ─── fact_sales

# Create Dimension:


In [0]:
#init

catalog_name = 'abhi_dwh_sql_based'
source_schema ='silver'
sink_schema   = 'gold'


#Clean start
spark.sql(f"DROP SCHEMA IF EXISTS {catalog_name}.{sink_schema} CASCADE")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{sink_schema}")


##  gold.dim_customers

In [0]:
%python
#Source Table
#| Table                  | Purpose             |
#| ---------------------- | ------------------- |
#| `silver.crm_cust_info` | Main customer data  |
#| `silver.erp_cust_az12` | Birthdate + gender  |
#| `silver.erp_loc_a101`  | Country information |
#

In [0]:
df_crm_cust=spark.read.table(f"{catalog_name}.{source_schema}.crm_cust_info")
df_erp_cust=spark.read.table(f"{catalog_name}.{source_schema}.erp_cust_az12")
df_erp_loc =spark.read.table(f"{catalog_name}.{source_schema}.erp_loc_a101")

In [0]:
#Tranformations
#| Transformation | Purpose               |
#| -------------- | --------------------- |
#| ROW_NUMBER()   | Create surrogate key  |
#| LEFT JOIN      | Merge ERP attributes  |
#| CASE           | Gender fallback logic |



In [0]:
print(f"df_crm_cust :{df_crm_cust.columns}")
print(f"df_erp_cust :{df_erp_cust.columns}")
print(f"df_erp_loc  :{df_erp_loc.columns}")

In [0]:
display(df_crm_cust.limit(2))
display(df_erp_cust.limit(2))
display(df_erp_loc.limit(2))

In [0]:
#join table 
df_dim_cust =(
    df_crm_cust.alias('ci')
    .join(df_erp_cust.alias('ca'), df_erp_cust.cid==df_crm_cust.cst_key, how='left')
    .join(df_erp_loc.alias('la'), df_erp_loc.CID==df_crm_cust.cst_key, how='left')
)

In [0]:
#Transform
from pyspark.sql.functions import row_number,when,coalesce
from pyspark.sql.window import Window
window_spec = Window.orderBy('cst_id')

df_dim_cust =df_dim_cust.select(
    row_number().over(window_spec).alias('customer_key'),
    col("cst_id").alias("customer_id"),
    col("cst_key").alias("customer_number"),
    col("cst_firstname").alias("first_name"),
    col("cst_lastname").alias("last_name"),
    col("cntry").alias("country"),
    col("cst_marital_status").alias("marital_status"),
    when(col("cst_gndr") != "n/a", col("cst_gndr"))
        .otherwise(coalesce(col("gen"), col("cst_gndr")))
        .alias("gender"),
    col("bdate").alias("birthdate"),
    col("cst_create_date").alias("create_date")
)

In [0]:
df_dim_cust.display()

## gold.dim_products

In [0]:
#Source Table 
#| Table                    | Purpose                    |
#| ------------------------ | -------------------------- |
#| `silver.crm_prd_info`    | Product details            |
#| `silver.erp_px_cat_g1v2` | Product category hierarchy |


In [0]:
#Transformation | Transformation       | Purpose                  |
#| -------------------- | ------------------------ |
#| `ROW_NUMBER()`       | Generate surrogate key   |
#| LEFT JOIN            | Add category attributes  |
#| `prd_end_dt IS NULL` | Keep only active records |


In [0]:
df_crm_prd_info=spark.read.table(f"{catalog_name}.{source_schema}.crm_prd_info")
df_erp_px_cat_g1v2=spark.read.table(f"{catalog_name}.{source_schema}.erp_px_cat_g1v2")
display(df_crm_prd_info.limit(2))
display(df_erp_px_cat_g1v2.limit(2))

In [0]:
#init 
from  pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window
window_spec = Window.orderBy('prd_id')

#join table 
df_dim_prod =df_crm_prd_info.alias('pi').join(df_erp_px_cat_g1v2.alias('pa'), df_erp_px_cat_g1v2.ID==df_crm_prd_info.prd_cat, how='left')
df_dim_prod=df_dim_prod.select(
    row_number().over(window_spec).alias('product_key'), #Surrogate key 
    col("prd_id").alias("product_id"),
    col("prd_key").alias("product_number"),
    col("prd_nm").alias("product_name"),
    col("prd_cat").alias("product_category"),
    col("CAT").alias("catagory"),
    col("MAINTENANCE").alias("maintenance"),
    col("prd_cost").alias("cost"),
    col("prd_line").alias("product_line"),
    col("prd_start_dt").alias("start_date")
)

df_dim_prod=df_dim_prod.filter(col("prd_end_dt").isNull())
df_dim_prod.display()

### gold.fact_sales

In [0]:
#Source Table 
#| Table                      | Purpose                |
#| -------------------------- | ---------------------- |
#| `silver.crm_sales_details` | Raw sales transactions |
#| `gold.dim_products`        | Product dimension      |
#| `gold.dim_customers`       | Customer dimension     |


In [0]:
df_sales = spark.table(f"{catalog_name}.{source_schema}.crm_sales_details")

In [0]:
#| Transformation      | Purpose                 |
#| ------------------- | ----------------------- |
#| LEFT JOIN products  | Map product_key         |
#| LEFT JOIN customers | Map customer_key        |
#| Rename columns      | Business-friendly names |


In [0]:
display(df_sales.limit(2))
display(df_dim_cust.limit(2))
display(df_dim_prod.limit(2))

In [0]:
#join 
#silver.crm_sales_details
#            │
#            │
#            ├── join → gold.dim_products
#            │
#            └── join → gold.dim_customers
#                        │
#                        ▼
#                  gold.fact_sales


df_fact_sales= (
    df_sales.
    join(df_dim_prod, df_dim_prod.product_number==df_sales.sls_prd_key, how='left').
    join(df_dim_cust, df_dim_cust.customer_id==df_sales.sls_cust_id, how='left')
)

In [0]:
df_fact_sales.display()

#Write  the gold dimensions and fact tables 

In [0]:
df_dim_customer.write.mode("overwrite").format("delta").saveAsTable(f"{catalog_name}.{sink_schema}.dim_customer")
print(f"Table {catalog_name}.{sink_schema}.dim_customer created with {df_dim_customer.count()} records")

df_dim_prod.write.mode("overwrite").format("delta").saveAsTable(f"{catalog_name}.{sink_schema}.dim_product")
print(f"Table {catalog_name}.{sink_schema}.dim_product created with {df_dim_prod.count()} records")

df_fact_sales.write.mode("overwrite").format("delta").saveAsTable(f"{catalog_name}.{sink_schema}.fact_sales")
print(f"Table {catalog_name}.{sink_schema}.fact_sales created with {df_fact_sales.count()} records")
